## Criando e testando a conexão

In [0]:
HOST = "aws-0-us-east-1.pooler.supabase.com"
PORT = "5432"
DATABASE = "postgres"
USER = "postgres.aqoqsumdougvmmpkooom"
PASS = "SenhaDoTrabalho"  # Vai ser usado só para o trabalho, então não há problema

url = f"jdbc:postgresql://{HOST}:{PORT}/{DATABASE}?sslmode=require"

props = {
    "user": USER,
    "password": PASS,
    "driver": "org.postgresql.Driver"
}

spark.read.jdbc(url, "games", properties=props).show(5)

+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+
| id|           name|         platform|release_date|metascore|user_score|           developer|           publisher|               genre|
+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+
|  1|          #BLUD|               PC|  2024-06-18|       73|       5.6|     Exit 73 Studios|    ['Humble Games']|    Action Adventure|
|  2|         #DRIVE|iOS (iPhone/iPad)|  2021-02-16|       69|       6.8|    Dariusz Pietrala|['PM Studios Inc....|         Auto Racing|
|  3|         #IDARB|         Xbox One|  2015-01-30|       77|       6.9|Other Ocean Inter...|['Other Ocean Int...|               Party|
|  4|#KILLALLZOMBIES|               PC|  2016-08-10|       53|       3.5|         Beatshapers|['Digerati Distri...|Top-Down Shoot-'E...|
|  5|#KILLALLZOMBIES|    PlayStation 4|  

## Lendo todas as tabelas e guardando em um DataFrame

In [0]:
df_games = spark.read.options(inferSchema=True, header=True).jdbc(url, "games", properties=props)
df_steam = spark.read.options(inferSchema=True, header=True).jdbc(url, "steam_metrics", properties=props)
df_sales = spark.read.options(inferSchema=True, header=True).jdbc(url, "regional_sales", properties=props)
df_hltob = spark.read.options(inferSchema=True, header=True).jdbc(url, "how_long_to_beat", properties=props)
df_links = spark.read.options(inferSchema=True, header=True).jdbc(url, "walkthrough_links", properties=props)

## Gravando os CSV na landing-zone

In [0]:
landing_path = '/Volumes/workspace/landing/dados'

df_games.coalesce(1).write.mode("overwrite").options(header=True, delimiter=",").csv(f"{landing_path}/temp_games")
df_steam.coalesce(1).write.mode("overwrite").options(header=True, delimiter=",").csv(f"{landing_path}/temp_steam_metrics")
df_sales.coalesce(1).write.mode("overwrite").options(header=True, delimiter=",").csv(f"{landing_path}/temp_regional_sales")
df_hltob.coalesce(1).write.mode("overwrite").options(header=True, delimiter=",").csv(f"{landing_path}/temp_how_long_to_beat")
df_links.coalesce(1).write.mode("overwrite").options(header=True, delimiter=",").csv(f"{landing_path}/temp_walkthrough_links")

## Movendo e renomeando os arquivos CSV

In [0]:
temp_paths = filter(lambda p: p.name.startswith("temp_"), dbutils.fs.ls(landing_path))

for path in temp_paths:
    name = path.name.replace("temp_", "")[:-1]

    part_file = [f.path for f in dbutils.fs.ls(path.path) if f.name.startswith("part-")][0]

    dbutils.fs.cp(part_file, f"{landing_path}/{name}.csv")
    dbutils.fs.rm(path.path, recurse=True)

display(dbutils.fs.ls(landing_path))